In [42]:
import brightway2 as bw
from premise import *
import os
import wurst
import time
import copy

In [43]:
startTime = time.time() # just to see how much time the code takes to run (this is the start)

In [44]:
bw.projects.set_current('Prospective LCA v5') # set current project

In [45]:
from premise_gwp import add_premise_gwp
add_premise_gwp()

Adding ('IPCC 2013', 'climate change', 'GTP 20a, incl. bio CO2')
Applying strategy: csv_restore_tuples
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: set_biosphere_type
Applying strategy: drop_unspecified_subcategories
Applying strategy: link_iterable_by_fields
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applied 8 strategies in 0.06 seconds
Wrote 1 LCIA methods with 217 characterization factors
Adding ('IPCC 2013', 'climate change', 'GWP 20a, incl. H')
Applying strategy: csv_restore_tuples
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: set_biosphere_type
Applying strategy: drop_unspecified_subcategories
Applying strategy: link_iterable_by_fields
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applied 8 strategies in 0.05 seconds
Wrote 1 LCIA methods with 216

In [46]:
bioSphereDBName = 'biosphere3'
bioSphereDB = bw.Database(bioSphereDBName) # import the biosphere database
ecoInventBase2020DBName = 'ecoinvent 3.8 cutoff image SSP2-Base 2020'
ecoInventBase2020DB = bw.Database(ecoInventBase2020DBName) # import the ecoinvent database (image base 2020)

In [47]:
# import base inventories from excel
excelImport = bw.ExcelImporter(os.path.join('..', 'Raw data', 'lci-Abhi image SSP2-Base 2020 ammonia and methanol.xlsx'))
excelImport.apply_strategies()
excelImport.match_database(ecoInventBase2020DBName, fields = ('name', 'reference product', 'unit', 'location')) # match flows from ecoinvent database
excelImport.match_database(bioSphereDBName, fields = ('name', 'unit', 'categories')) # match flows from biosphere database
excelImport.match_database(fields = ('name', 'reference product', 'unit', 'location')) # match flows from my database
excelImport.statistics()

Extracted 2 worksheets in 0.01 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 0.07 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
16 datasets
147 exchanges
0 unlinked exchang

(16, 147, 0)

In [48]:
# modify inventories to substitute 'reference product' with 'product' in all the exchanges
for ds in excelImport:
    for exchange in ds['exchanges']:
        if 'reference product' in exchange:
            exchange['product'] = exchange.pop('reference product')

In [49]:
lciBase2020DBName = 'lci-Abhi image SSP2-Base 2020 ammonia and methanol'

In [50]:
newLocations = {'BR' : 'Brazil',
                'CA' : 'Canada',
                'PL' : 'Central Europe',
                'CN' : 'China',
                'ET' : 'Eastern Africa',
                'IN' : 'India',
                'ID' : 'Indonesia',
                'JP' : 'Japan',
                'KR' : 'Korea',
                'IR' : 'Middle East',
                'MX' : 'Mexico',
                'EG' : 'Northern Africa',
                'AU' : 'Oceania',
                'GT' : 'Rest of Central America',
                'BW' : 'Rest of Southern Africa',
                'CL' : 'Rest of Southern America',
                'PK' : 'Rest of Southern Asia',
                'RU' : 'Russia',
                'ZA' : 'South Africa',
                'PH' : 'South Eastern Asia',
                'UZ' : 'Central Asia',
                'TR' : 'Turkey',
                'UA' : 'Ukraine',
                'US' : 'United States of America',
                'NG' : 'Western Africa',
                'RER' : 'Western Europe'}

In [51]:
globalDB = excelImport.data # add all inventories
ecoInventBase2020DBDict = [ds.as_dict() for ds in ecoInventBase2020DB] # convert ecoinvent database to dictionary
bioSphereDBDict = [ds.as_dict() for ds in bioSphereDB] # convert biosphere database to dictionary # maybe not needed?

DSToRegionalize = globalDB

regionalizedDS = []
for ds in DSToRegionalize:
    for loc in newLocations:
        dsCopy = wurst.transformations.geo.copy_to_new_location(ds, loc)
        regionalizedDS.append(dsCopy)        

# Relink technosphere exchanges to the new locations
for ds in regionalizedDS:
    for exchange in ds['exchanges']:
        if exchange['type'] == 'technosphere':
            if exchange['database'] == ecoInventBase2020DBName:
                dsOutput = [dsInstance for dsInstance in ecoInventBase2020DBDict if exchange['name'] == dsInstance['name'] 
                                and exchange['product'] == dsInstance['reference product'] 
                                and ds['location'] == dsInstance['location']]
                if not dsOutput and 'market group for electricity' in exchange['name']:
                    exchange['name'] = exchange['name'].replace('group ', '')
                    dsOutput = [dsInstance for dsInstance in ecoInventBase2020DBDict if exchange['name'] == dsInstance['name'] 
                                and exchange['product'] == dsInstance['reference product'] 
                                and ds['location'] == dsInstance['location']]
            elif exchange['database'] == lciBase2020DBName:
                dsOutput = [dsInstance for dsInstance in regionalizedDS if exchange['name'] == dsInstance['name']
                                and ds['location'] == dsInstance['location']]
            if dsOutput:
                    exchange.update({'location' : dsOutput[0]['location']})

In [52]:
db = globalDB + regionalizedDS
DBLinked = copy.deepcopy(db)

production = lambda x : x['type'] == 'production'
technosphere = lambda x : x['type'] == 'technosphere'
biosphere = lambda x : x['type'] == 'biosphere'

# linking exchanges to database inventories
for ds in DBLinked:
    for exchange in filter(production, ds['exchanges']):
        exchange.update({'input' : (ds['database'], ds['code'])})
    for exchange in filter(technosphere, ds['exchanges']):
        try:
            exchangeLink = wurst.get_one(db + ecoInventBase2020DBDict,
                                         wurst.equals('name', exchange['name']),
                                         wurst.equals('reference product', exchange['product']),
                                         wurst.equals('location', exchange['location']))
            exchange.update({'input' : (exchangeLink['database'], exchangeLink['code'])})
        except Exception:
            print(exchange['name'], exchange['product'], exchange['location'])
            raise
    # biosphere links maybe not needed
    for exchange in filter(biosphere, ds['exchanges']):
        if 'input' not in exchange:
            try:
                exchangeLink = [ds['code'] for ds in bioSphereDBDict if ds['name'] == exchange['name'] and
                                                                        ds['unit'] == exchange['unit'] and
                                                                        ds['categories'] == exchange['categories']][0]
                exchange.update({'input' : (bioSphereDBName, exchangeLink)})
            except Exception:
                print(exchange['name'], exchange['unit'], exchange['categories'])
                raise

In [53]:
if lciBase2020DBName in bw.databases:
    del bw.databases[lciBase2020DBName]  
wurst.write_brightway2_database(DBLinked, lciBase2020DBName) # write database

432 datasets
3969 exchanges
0 unlinked exchanges
  
Title: Writing activities to SQLite3 database:
  Started: 12/15/2023 15:19:33
  Finished: 12/15/2023 15:19:33
  Total time elapsed: 00:00:00
  CPU %: 96.70
  Memory %: 7.60
Created database: lci-Abhi image SSP2-Base 2020 ammonia and methanol


In [54]:
lciBase2020DB = wurst.extract_brightway2_databases(lciBase2020DBName)

Getting activity data


100%|██████████| 432/432 [00:00<00:00, 287655.08it/s]


Adding exchange data to activities


100%|██████████| 3969/3969 [00:00<00:00, 86417.87it/s]


Filling out exchange data


100%|██████████| 432/432 [00:00<00:00, 8444.75it/s]


In [55]:
databaseNames = bw.databases
ecoInventDBNames = []
premiseDBNames = []
for databaseName in databaseNames:
    if '2020' not in databaseName and 'SSP2' in databaseName and 'Abhi' not in databaseName:
        ecoInventDBNames.append(databaseName)
        premiseDBNames.append(databaseName.replace('ecoinvent 3.8 cutoff ', ''))
ecoInventDBNames.sort()
premiseDBNames.sort()

In [56]:
for premiseDBName in premiseDBNames:
    print('linking database ' + premiseDBName + '...')
    premiseDBDict = [ds.as_dict() for ds in bw.Database('ecoinvent 3.8 cutoff ' + premiseDBName)]

    DBLinked = copy.deepcopy(lciBase2020DB)

    production = lambda x : x['type'] == 'production'  
    technosphere = lambda x : x['type'] == 'technosphere'
    biosphere = lambda x : x['type'] == 'biosphere'

    for ds in DBLinked:
        for exchange in filter(technosphere, ds['exchanges']):
            try:
                exchangeLink = wurst.get_one(lciBase2020DB + premiseDBDict,
                                            wurst.equals('name', exchange['name']),
                                            wurst.equals('reference product', exchange['product']),
                                            wurst.equals('location', exchange['location'])
                                            )
                exchange.update({'input' : (exchangeLink['database'], exchangeLink['code'])})
                exchange.update({'database' : exchangeLink['database']})
            except Exception:
                print(exchange['name'], exchange['reference product'], exchange['location'])
                raise
        for exchange in filter(biosphere, ds['exchanges']):
            if 'input' not in exchange:
                try:
                    exchangeLink = [ds['code'] for ef in bioSphereDBDict if ds['name'] == exchange['name'] and 
                                                                            ds['unit'] == exchange['unit'] and 
                                                                            ds['categories'] == exchange['categories']][0]
                    exchange.update({'input': ('biosphere3', exchangeLink)})   
                except Exception:
                    print(exchange['name'], exchange['unit'], exchange['categories'])
                    raise
    dbName = 'lci-Abhi ' + premiseDBName + ' ammonia and methanol'
    if dbName in bw.databases:
        del bw.databases[dbName]
    wurst.write_brightway2_database(DBLinked, dbName)
    print(premiseDBName + ' linking and writing successful!')

linking database image SSP2-Base 2030...
432 datasets
3969 exchanges
0 unlinked exchanges
  
Title: Writing activities to SQLite3 database:
  Started: 12/15/2023 15:20:56
  Finished: 12/15/2023 15:20:56
  Total time elapsed: 00:00:00
  CPU %: 99.80
  Memory %: 7.65
Created database: lci-Abhi image SSP2-Base 2030 ammonia and methanol
image SSP2-Base 2030 linking and writing successful!
linking database image SSP2-Base 2040...
432 datasets
3969 exchanges
0 unlinked exchanges
  
Title: Writing activities to SQLite3 database:
  Started: 12/15/2023 15:22:12
  Finished: 12/15/2023 15:22:13
  Total time elapsed: 00:00:00
  CPU %: 99.00
  Memory %: 9.06
Created database: lci-Abhi image SSP2-Base 2040 ammonia and methanol
image SSP2-Base 2040 linking and writing successful!
linking database image SSP2-Base 2050...
432 datasets
3969 exchanges
0 unlinked exchanges
  
Title: Writing activities to SQLite3 database:
  Started: 12/15/2023 15:23:38
  Finished: 12/15/2023 15:23:38
  Total time elapsed:

In [57]:
endTime = time.time() # end time
elapsedTime = endTime - startTime # calculate elapsed time
print(f'Elapsed time: {elapsedTime/3600:.2f} hours') 

Elapsed time: 0.23 hours
